# ShopLens — Autoencoder Training (Epics 5-6)

Trains the movement anomaly autoencoder on normal trajectories, tracks experiments in MLflow,
and exports the best model to `backend/models/autoencoder.pth`.

**Run order:** installs → setup → data → features → train → threshold → evaluate → export.
The backend only ever *loads* the exported artifacts — all training lives here.

In [ ]:
%pip install 'numpy<2.0' torch scikit-learn joblib mlflow matplotlib
%pip install opencv-python-headless scipy

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler

DRIVE_DIR = Path('/content/drive/MyDrive/shoplens')
TRAJECTORIES_JSON = DRIVE_DIR / 'trajectories.json'
EXPORT_PATH = DRIVE_DIR / 'autoencoder.pth'

mlflow.set_experiment('shoplens-autoencoder')
torch.manual_seed(42)


## Load trajectories (Epic 2 handoff)
`{track_id: [{frame, cx, cy}, ...]}` — same format `tracking.ipynb` saved.

In [ ]:
with open(TRAJECTORIES_JSON) as f:
    trajectories = json.load(f)

print(f'{len(trajectories)} tracks loaded')


## Feature extraction
Feature vector per person per zone-visit: `(zone_id_index, dwell_time, velocity, direction_change)`.
Must match `backend/cv/anomaly.py::extract_trajectory_features` exactly.

In [ ]:
FRAME_INTERVAL_SEC = 5 / 30  # sampled every 5th frame at 30fps source

# TODO(epic-5 day 2): port compute_zone_visits + per-point deltas here.
# Zone ids must be encoded with a stable index map shared with inference.
features_raw = None  # np.ndarray of shape (n_visits, 4)
assert features_raw is not None, 'implement extraction first'


In [ ]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_raw)

# unsupervised: everything is treated as normal during training;
# held-out split exists only for honest reconstruction-error numbers
rng = np.random.default_rng(42)
perm = rng.permutation(len(features_scaled))
split = int(0.8 * len(perm))
train_x = features_scaled[perm[:split]]
test_x = features_scaled[perm[split:]]


In [ ]:
class TrajectoryAutoencoder(torch.nn.Module):
    def __init__(self, dim=4):
        super().__init__()
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(dim, 8), torch.nn.ReLU(),
            torch.nn.Linear(8, 4), torch.nn.ReLU(),
            torch.nn.Linear(4, 2),
        )
        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(2, 4), torch.nn.ReLU(),
            torch.nn.Linear(4, 8), torch.nn.ReLU(),
            torch.nn.Linear(8, dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


model = TrajectoryAutoencoder()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()


## Train with MLflow tracking
Log hyperparameters and per-epoch loss so runs are comparable in the MLflow UI.

In [ ]:
EPOCHS = 100
LR = 1e-3

run = mlflow.start_run()
mlflow.log_params({'epochs': EPOCHS, 'lr': LR, 'train_rows': len(train_x)})

x_train = torch.tensor(train_x, dtype=torch.float32)
for epoch in range(EPOCHS):
    opt.zero_grad()
    recon = model(x_train)
    loss = loss_fn(recon, x_train)
    loss.backward()
    opt.step()
    if epoch % 10 == 0:
        mlflow.log_metric('train_loss', float(loss), step=epoch)
        print(f'epoch {epoch:4d}  loss {float(loss):.5f}')

mlflow.log_metric('final_loss', float(loss))


## Threshold: 95th percentile of reconstruction error

In [ ]:
with torch.no_grad():
    x_test = torch.tensor(test_x, dtype=torch.float32)
    errors = ((model(x_test) - x_test) ** 2).mean(dim=1).numpy()

threshold = float(np.percentile(errors, 95))
mlflow.log_metric('anomaly_threshold', threshold)
print(f'threshold = {threshold:.5f}')

plt.hist(errors, bins=50)
plt.axvline(threshold, color='r', label='p95 threshold')
plt.legend()
plt.show()


In [ ]:
# TODO(epic-5 day 4): overlay flagged vs normal visit positions on an actual store frame
# and eyeball whether the flagged ones look genuinely unusual before trusting the threshold.


In [ ]:
torch.save(model.state_dict(), EXPORT_PATH)
print(f'model exported to {EXPORT_PATH}')
# TODO(epic-5 day 5): compare runs in the MLflow UI, then copy the winning
# autoencoder.pth into backend/models/ alongside its scaler joblib file.


## Notes
- Compare runs by final_loss and anomaly_threshold in the MLflow UI (`mlflow ui`).
- Record the chosen run's params in the technical report (Epic 9).
- False positive rate on known-normal behaviour goes here once labels exist (Epic 6 Day 5).